# Task 3 – Merge and quality checks
**Who:** Joud & Maram  
**Input:** `food_clean.csv` (Task2_Cleaning), `joud_features.csv` (Task2_Joud_Violations), `maram_features.csv` (Task2_Maram_History). Run those three notebooks first, on the same computer.  
**Output:** `food_final.csv`, the final dataset for the model.

## 1. Merge the three files

In [ ]:
import pandas as pd

We only need pandas in this notebook (no plots).

In [ ]:
clean = pd.read_csv('food_clean.csv')
print(clean.shape)

The clean file: one row per real inspection, with the original columns plus Business_ID, First_Seen_Date, Year, Fail and Period. (Write here the shape.)

In [ ]:
joud = pd.read_csv('joud_features.csv')
print(joud.shape)

Joud's violation features (counts, severity levels, topics, previous-visit columns). (Write here the shape.)

In [ ]:
maram = pd.read_csv('maram_features.csv')
print(maram.shape)

Maram's history and calendar features. (Write here the shape.)

In [ ]:
print(clean['Inspection ID'].duplicated().sum(), joud['Inspection ID'].duplicated().sum(), maram['Inspection ID'].duplicated().sum())

Before merging we check that `Inspection ID` is unique in each file, so every inspection appears only once. All three numbers should be 0.

In [ ]:
final = clean.merge(joud, on='Inspection ID', how='inner', validate='one_to_one')
final = final.merge(maram, on='Inspection ID', how='inner', validate='one_to_one')
print(final.shape)

We join the three files on `Inspection ID` (`merge` is new for us: it puts the columns of two tables side by side, matching rows with the same ID, like VLOOKUP in a spreadsheet). `validate='one_to_one'` stops with an error if an ID appears twice. The number of rows must stay the same as in the clean file. (Write here the shape.)

## 2. Quality checks

In [ ]:
print(len(final) == len(clean) == len(joud) == len(maram))

Quality check 1: no rows were lost or added by the merge (should print True).

In [ ]:
final.isnull().sum()[final.isnull().sum() > 0]

Quality check 2: the only columns with missing values should be descriptive columns we do not use (AKA Name, City, State, Zip, Latitude, Longitude, Location) and Violations (empty = no violations). All feature columns must have 0 missing values. (Write here what you see.)

In [ ]:
final['Inspection Date'] = pd.to_datetime(final['Inspection Date'])
print(final['Inspection Date'].min(), final['Inspection Date'].max())

Quality check 3: the dates are between 1 January 2015 and 31 December 2022.

In [ ]:
final['Results'].value_counts()

Quality check 4: only the three real results are left.

In [ ]:
final['Fail'].mean().round(3)

The share of failed inspections in the final dataset (the target). (Write here the share. About 1 in 5 inspections fails, so the classes are unbalanced; this matters for the model.)

In [ ]:
final['Is_Reinspection'] = final['Is_Reinspection'].astype(int)
final.to_csv('food_final.csv', index=False)
print(final.shape)

We save the final dataset. `Is_Reinspection` is stored as 0/1 like the other yes/no columns. The meaning of every column is in `README.md`.